# 关于本 Notebook

本 Notebook 给出了一套构建生产级 AI 智能体（AI Agents）的蓝图，重点解决从“在我的机器上能跑”到“在大规模生产环境中可靠运行”之间的关键差距。

**问题：** 智能体在生产环境中经常会静默失败（silent failure）。当 LLM 输出出现轻微变化、JSON 解析失败，或不同提供商返回不一致的数据类型（例如本应是整数却返回字符串）时，往往会导致大量时间被浪费在脆弱代码的调试上。

**解决方案：** 使用 **Pydantic 模型（Pydantic models）** 作为技术契约（technical contracts），在系统边界明确规定数据的精确结构。这样可以阻止无效状态继续传播，简化模型提供商切换，并让系统具备更强的自恢复能力。

---

## 面向生产环境的技术支柱

### 1. 确定性的策略管理（Deterministic Policy Management）
为了保证一次运行从开始到结束保持稳定，验证策略必须在**运行开始时固定（pin）**。
* **策略哈希（Policy Hashing）：** 对当前生效的配置进行哈希，并把结果保存到运行状态中。
* **一致性（Consistency）：** 即使部署过程中全局配置发生变化，已经开始的运行仍继续遵循初始化时的规则。
* **审计轨迹（Audit Trails）：** 每条数据库记录都包含 `policy_hash`，开发者可以据此准确追溯生成该数据时使用了哪套验证规则。

### 2. 面向性能优化的验证
验证本身不应成为性能瓶颈。
* **模型缓存（Model Caching）：** Pydantic 模型针对每个 `(mode, policy_hash)` 组合只构建一次并缓存，避免在“热路径（hot paths）”中反复生成类。
* **直接传递字典：** 节点把原始数据直接传给工厂函数，避免在执行图内部增加不必要的解析开销。

### 3. 显式状态管理（Explicit State Management）
与其通过错误次数推断是否成功，不如使用**布尔验证标志**（例如 `prompt_ok`、`image_ok`）。
* **明确的失败模式：** 路由逻辑直接检查这些标志。
* **易读日志：** 重试逻辑先递增计数，再判断是否超过上限，因此日志可以清楚展示“Attempt 1”到“Attempt N”。

### 4. 边界分离（Boundary Separation）
严格区分**工具结果（Tool Results）**与**数据库记录（Database Records）**。
* **契约独立性：** 验证工具输出的 Schema 可以独立于永久存储所使用的 Schema 演进。
* **类型安全契约：** 数据库记录接收的是已经验证过的 `BaseModel` 实例，而不是原始字典或 `Any` 类型。

---

## 架构设计模式

实现采用**三块式结构（Three-Block Structure）**：

| 组件 | 职责 |
| :--- | :--- |
| **Governance Config** | 使用 OmegaConf 集中管理策略，并支持 Strict / Lenient 等模式切换。 |
| **Contracts & Gates** | 通过缓存的 Pydantic 工厂构建防御性边界，验证不可信数据。 |
| **Graph Wiring** | 使用 LangGraph 工作流管理状态持久化、检查点（checkpointer）和重试逻辑。 |

---

## Pydantic 与 Instructor：策略层面对比

Pydantic 主要保护图内部的系统边界，而 **Instructor** 则把恢复机制进一步推近到生成源头。

| 特性 | LangGraph + Pydantic | Instructor |
| :--- | :--- | :--- |
| **恢复逻辑** | 通过 refine 节点把恢复放在**图（Graph）**中。 | 通过自动重试把恢复放在**客户端（Client）**中。 |
| **可见性** | 每一次重试都是 Trace 中独立的节点。 | 重试在 LLM 调用内部完成。 |
| **复杂度** | 较高，需要条件路由和循环。 | 较低，从生成到下一步可以保持线性流程。 |
| **更适合** | 复杂多智能体系统以及审计要求较高的任务。 | 结构化数据抽取与简单的“修复后重试”循环。 |

---

## 关键实现细节

本 Notebook 演示了几个对生产环境非常重要的实现细节：
1. **稳定的 Schema 哈希（Stable Schema Hashing）：** 动态生成的模型使用显式 title，确保 JSON Schema 引用在不同进程之间保持稳定。
2. **积极归一化（Aggressive Normalization）：** 在哈希前，对验证关键词和策略值排序并转成小写，避免无意义的缓存失效。
3. **可追溯性（Traceability）：** `run_id` 把每条数据库记录关联到具体的 LangGraph `thread_id` 与 checkpoint 状态。

In [1]:
%%capture
!pip install langgraph langchain-core langchain-openai typing-extensions instructor


In [2]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

# 导入依赖

In [15]:
from __future__ import annotations

# Python 标准库
import json
import hashlib
from datetime import datetime, timezone
from operator import add
from typing import Any, Annotated, Literal, Optional, List
from typing_extensions import TypedDict

# LangChain / LangGraph 相关依赖
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver

# 第三方依赖
from omegaconf import OmegaConf, DictConfig
from pydantic import BaseModel, Field, ValidationError, field_validator
import instructor


## 问题：静默失败（Silent Failures）

先看一个常见场景：一个处理用户数据的智能体。如果不使用 Pydantic，类型不匹配可能不会立即报错，而是静默地进入后续流程，或者在更深层的位置触发晦涩错误。

下面展示一种典型的静默失败：字符串（String）与整数（Integer）的混用。数据库期望整数，但 LLM 返回了字符串。

In [4]:
def process_user_data_brittle(data: dict):
    """没有验证时，这段逻辑可能静默失败，或以不可预测的方式崩溃。"""
    user_id = data.get("user_id")  # 既可能是 "123"（字符串），也可能是 123（整数）
    age = data.get("age")  # 既可能是 "25"，也可能是 25

    # 在预发布环境中可能正常，因为测试数据恰好是正确类型
    # 但当生产环境中的 LLM 返回字符串时就可能失败
    result = user_id * 2  # 如果 user_id 是 "123"，行为就与整数完全不同
    database_query = f"SELECT * FROM users WHERE age > {age}"  # 如果 age 是未经约束的字符串，还可能引入 SQL 注入风险

    return {"processed_id": result, "query": database_query}

# 模拟 LLM 返回不一致数据类型时的情况
llm_response_1 = {"user_id": 123, "age": 25}  # ✅ 可正常工作（预发布）
llm_response_2 = {"user_id": "123", "age": "25"}  # ❌ 在生产环境中可能静默地产生错误结果

print("Response 1 (staging):")
try:
    result = process_user_data_brittle(llm_response_1)
    print(f"  Success: {result}")
except Exception as e:
    print(f"  Error: {e}")

print("\nResponse 2 (production):")
try:
    result = process_user_data_brittle(llm_response_2)
    print(f"  Success: {result}")
except Exception as e:
    print(f"  Error: {type(e).__name__}: {e}")

Response 1 (staging):
  Success: {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25'}

Response 2 (production):
  Success: {'processed_id': '123123', 'query': 'SELECT * FROM users WHERE age > 25'}


## 解决方案：Pydantic 验证（Pydantic Validation）

Pydantic 会在系统边界**立即**捕获这些错误，而不是让错误继续向系统内部传播。

In [5]:
# 精确定义数据结构
class UserData(BaseModel):
    """技术规范：精确定义系统期望接收的数据。"""
    user_id: int = Field(..., description="User ID must be an integer")
    age: int = Field(..., ge=0, le=150, description="Age must be between 0 and 150")
    email: Optional[str] = Field(None, description="Optional email address")

    class Config:
        # 在可行时自动把字符串转换为整数
        # 用这种方式优雅吸收不同提供商之间的差异
        str_strip_whitespace = True

def process_user_data_safe(data: dict):
    """使用 Pydantic 后，验证会发生在系统边界。"""
    try:
        # 在这里完成验证：数据错误时立即失败（fail fast）
        user = UserData(**data)

        # 通过验证后，就可以信任这些数据类型
        result = user.user_id * 2
        database_query = f"SELECT * FROM users WHERE age > {user.age}"

        return {"processed_id": result, "query": database_query, "validated": user}
    except ValidationError as e:
        return {"error": "Validation failed", "details": str(e)}

# 使用相同的测试用例
llm_response_1 = {"user_id": 123, "age": 25}
llm_response_2 = {"user_id": "123", "age": "25"}  # 字符串：Pydantic 会完成转换
llm_response_3 = {"user_id": "abc", "age": "25"}  # 无效数据：Pydantic 会立即捕获

print("Response 1 (valid ints):")
result = process_user_data_safe(llm_response_1)
print(f"  {result}")

print("\nResponse 2 (string numbers - Pydantic converts):")
result = process_user_data_safe(llm_response_2)
print(f"  {result}")

print("\nResponse 3 (invalid data - Pydantic catches):")
result = process_user_data_safe(llm_response_3)
print(f"  {result}")

Response 1 (valid ints):
  {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25', 'validated': UserData(user_id=123, age=25, email=None)}

Response 2 (string numbers - Pydantic converts):
  {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25', 'validated': UserData(user_id=123, age=25, email=None)}

Response 3 (invalid data - Pydantic catches):
  {'error': 'Validation failed', 'details': "1 validation error for UserData\nuser_id\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing"}


/tmp/ipykernel_1461/2645436559.py:2: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class UserData(BaseModel):


## 模型提供商差异：真实世界的问题

不同 LLM 提供商返回的 JSON 结构往往会有细微差异。如果没有验证层，就需要针对每个提供商编写单独的解析逻辑。

In [6]:
# 模拟不同模型提供商返回的响应
class ProviderResponse(BaseModel):
    """可跨不同提供商复用的统一模型。"""
    task: str
    priority: int = Field(ge=1, le=5)
    tags: List[str] = Field(default_factory=list)
    metadata: dict = Field(default_factory=dict)

# Provider A：返回一种结构
provider_a_response = {
    "task": "Write blog post",
    "priority": "3",  # 返回字符串而不是整数
    "tags": ["ai", "agents"],
    "metadata": {"source": "user"}
}

# Provider B：返回字段命名不同的扁平结构
provider_b_response = {
    "task": "Write blog post",
    "priority": 3,
    "tags": "ai,agents",  # 返回逗号分隔字符串，而不是列表
    "extra": {"source": "user"}  # 使用了不同的字段名
}

# Provider C：JSON 合法，但缺少部分字段
provider_c_response = {
    "task": "Write blog post",
    "priority": 3
    # 缺少 tags 和 metadata
}

def parse_provider_response_brittle(data: dict, provider: str):
    """如果没有 Pydantic，就需要为不同提供商分别编写解析逻辑。"""
    if provider == "A":
        priority = int(data.get("priority", 0))
        tags = data.get("tags", [])
    elif provider == "B":
        priority = data.get("priority", 0)
        tags = data.get("tags", "").split(",") if isinstance(data.get("tags"), str) else data.get("tags", [])
    else:
        priority = data.get("priority", 0)
        tags = data.get("tags", [])

    return {"task": data.get("task"), "priority": priority, "tags": tags}

def parse_provider_response_safe(data: dict):
    """使用 Pydantic 后，可以用同一个模型处理不同提供商。"""
    # Pydantic 会自动完成：
    # - 把字符串 "3" 转成整数 3
    # - 处理缺失字段（使用默认值）
    # - 校验类型与约束
    try:
        # 处理 Provider B 返回的逗号分隔 tags
        if isinstance(data.get("tags"), str):
            data["tags"] = [tag.strip() for tag in data["tags"].split(",") if tag.strip()]

        # 处理 Provider B 不同的 metadata 字段名
        if "extra" in data and "metadata" not in data:
            data["metadata"] = data.pop("extra")

        return ProviderResponse(**data)
    except ValidationError as e:
        return {"error": str(e)}

print("Provider A (string priority):")
result = parse_provider_response_safe(provider_a_response)
print(f"  {result}")

print("\nProvider B (comma-separated tags, different key):")
result = parse_provider_response_safe(provider_b_response)
print(f"  {result}")

print("\nProvider C (missing fields):")
result = parse_provider_response_safe(provider_c_response)
print(f"  {result}")

Provider A (string priority):
  task='Write blog post' priority=3 tags=['ai', 'agents'] metadata={'source': 'user'}

Provider B (comma-separated tags, different key):
  task='Write blog post' priority=3 tags=['ai', 'agents'] metadata={'source': 'user'}

Provider C (missing fields):
  task='Write blog post' priority=3 tags=[] metadata={}


## 为什么 Pydantic 对智能体很重要

1. **防止静默失败：** 在系统边界捕获类型不匹配，而不是等错误深入代码后才暴露
2. **提供商一致性：** 一个模型可以吸收不同提供商之间的常见差异，例如字符串与整数、不同字段名等
3. **类型安全（Type Safety）：** IDE 和类型检查器能够理解你的数据结构
4. **自描述（Self-Documenting）：** 模型本身就是规范，不需要另外维护一套文档
5. **LangGraph 集成：** 验证后的 State 能避免无效数据在智能体步骤之间传播

## 最佳实践

- **在边界定义模型：** 在数据进入系统的位置进行建模，例如 LLM 响应、API 调用和用户输入
- **使用 Field 约束：** 通过 `ge`、`le`、`pattern` 等参数表达验证规则
- **吸收提供商差异：** 利用 Pydantic 的类型强制转换（type coercion）处理常见差异
- **快速失败（Fail Fast）：** 让验证错误尽早暴露，不要在深层逻辑中尝试“修补”无效数据

# 整合所有概念

## 治理配置（Governance Config）与策略选择

In [7]:
GOVERNANCE_CONFIG = """
# 运行模式："strict"、"moderate" 或 "lenient"
mode: "strict"

# 各模式对应的策略
policies:
  strict:
    max_retries: 3
    prompt:
      min_length: 50
      required_keywords: ["pydantic", "validation", "contract"]
      forbid_markdown: true  # 真实约束：Prompt 中不允许代码块
      require_style_tag: true  # 必须包含风格描述词
      size_px_min: 512
      size_px_max: 1536
    image_output:
      allowed_mime: ["image/png"]
      max_bytes: 2000000
    db:
      serialize_mode: "json"
      schema_version: 1

  moderate:
    max_retries: 2
    prompt:
      min_length: 30
      required_keywords: ["pydantic"]
      forbid_markdown: false
      require_style_tag: false
      size_px_min: 256
      size_px_max: 2048
    image_output:
      allowed_mime: ["image/png", "image/jpeg"]
      max_bytes: 5000000
    db:
      serialize_mode: "json"
      schema_version: 1

  lenient:
    max_retries: 1
    prompt:
      min_length: 20
      required_keywords: []
      forbid_markdown: false
      require_style_tag: false
      size_px_min: 256
      size_px_max: 4096
    image_output:
      allowed_mime: ["image/png", "image/jpeg", "image/webp"]
      max_bytes: 10000000
    db:
      serialize_mode: "json"
      schema_version: 1
"""

cfg: DictConfig = OmegaConf.create(GOVERNANCE_CONFIG)

def get_active_policy(cfg: DictConfig) -> DictConfig:
    """根据当前模式获取生效策略。"""
    mode = str(cfg.mode)
    return cfg.policies[mode]


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def hash_policy(policy_dict: dict[str, Any]) -> str:
    """为策略生成确定性哈希，用于版本追踪。"""
    canonical = json.dumps(policy_dict, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return sha256_text(canonical)[:16]


## 契约与验证门（Contracts and Gates，带缓存）

In [8]:
# 按 (mode, policy_hash) 缓存 Pydantic 模型，避免复用过期验证器
# 返回：(ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord)
_model_cache: dict[tuple[str, str], tuple[type[BaseModel], type[BaseModel], type[BaseModel]]] = {}


def build_image_prompt_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """工厂函数：根据策略字典构建 Pydantic 模型，并按 mode 与 policy_hash 缓存。"""
    prompt_cfg = policy_dict["prompt"]
    min_len = int(prompt_cfg["min_length"])
    required = [str(x).lower() for x in prompt_cfg.get("required_keywords", [])]
    size_min = int(prompt_cfg["size_px_min"])
    size_max = int(prompt_cfg["size_px_max"])
    forbid_md = bool(prompt_cfg.get("forbid_markdown", False))
    require_style = bool(prompt_cfg.get("require_style_tag", False))

    class ImagePrompt(BaseModel):
        model_config = {"title": f"ImagePrompt_{mode}_{policy_hash}"}  # 保持 Schema 引用稳定

        prompt: str = Field(..., min_length=min_len)
        size_px: int = Field(default=1024, ge=size_min, le=size_max)

        @field_validator("prompt")
        @classmethod
        def enforce_keywords(cls, v: str) -> str:
            if required:
                low = v.lower()
                missing = [k for k in required if k not in low]
                if missing:
                    raise ValueError(f"Prompt missing required keywords: {missing}")
            return v

        @field_validator("prompt")
        @classmethod
        def forbid_markdown_blocks(cls, v: str) -> str:
            # 这里只检查代码块（三个反引号），不限制行内格式
            if forbid_md and "```" in v:
                raise ValueError("Prompt must not contain markdown code blocks")
            return v

        @field_validator("prompt")
        @classmethod
        def require_style_descriptor(cls, v: str) -> str:
            if require_style:
                style_tags = ["minimal", "vector", "illustration", "photo", "diagram"]
                low = v.lower()
                if not any(tag in low for tag in style_tags):
                    raise ValueError(f"Prompt must include a style tag: {style_tags}")
            return v

    return ImagePrompt


def build_generated_image_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """工厂函数：根据策略字典构建 Pydantic 模型，并按 mode 与 policy_hash 缓存。"""
    image_cfg = policy_dict["image_output"]
    allowed_mime = {str(x) for x in image_cfg["allowed_mime"]}
    max_bytes = int(image_cfg["max_bytes"])

    class GeneratedImageToolResult(BaseModel):
        """工具结果契约：验证图像生成工具的输出。"""
        model_config = {"title": f"GeneratedImageToolResult_{mode}_{policy_hash}"}  # 保持 Schema 引用稳定

        asset_id: str = Field(..., min_length=8)
        mime_type: str
        bytes_size: int = Field(..., ge=1, le=max_bytes)
        width_px: int = Field(..., ge=1)
        height_px: int = Field(..., ge=1)

        @field_validator("mime_type")
        @classmethod
        def validate_mime(cls, v: str) -> str:
            if v not in allowed_mime:
                raise ValueError(f"mime_type must be one of {sorted(allowed_mime)}")
            return v

    return GeneratedImageToolResult


def build_generated_image_record_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """工厂函数：构建用于图像存储的数据库记录模型，并与工具结果契约分离。"""
    image_cfg = policy_dict["image_output"]
    allowed_mime = {str(x) for x in image_cfg["allowed_mime"]}
    max_bytes = int(image_cfg["max_bytes"])

    class GeneratedImageRecord(BaseModel):
        """数据库记录契约：验证持久化的图像字段，并可独立于工具契约演进。"""
        model_config = {"title": f"GeneratedImageRecord_{mode}_{policy_hash}"}  # 保持 Schema 引用稳定

        asset_id: str = Field(..., min_length=8)
        mime_type: str
        bytes_size: int = Field(..., ge=1, le=max_bytes)
        width_px: int = Field(..., ge=1)
        height_px: int = Field(..., ge=1)

        @field_validator("mime_type")
        @classmethod
        def validate_mime(cls, v: str) -> str:
            if v not in allowed_mime:
                raise ValueError(f"mime_type must be one of {sorted(allowed_mime)}")
            return v

    return GeneratedImageRecord


def get_models_for_policy(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """获取某套策略对应的缓存模型。缓存键包含 policy_hash，以避免复用过期验证器。

    返回：(ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord)
    """
    cache_key = (mode, policy_hash)
    if cache_key not in _model_cache:
        _model_cache[cache_key] = (
            build_image_prompt_model(policy_dict, mode, policy_hash),
            build_generated_image_model(policy_dict, mode, policy_hash),  # 工具结果契约
            build_generated_image_record_model(policy_dict, mode, policy_hash),  # 数据库记录契约
        )
    return _model_cache[cache_key]


class MediaRecord(BaseModel):
    """带 Schema 版本控制与审计轨迹的数据库记录契约。

    注意：image 字段接收任意 BaseModel 实例（GeneratedImageRecord 会动态构建
    于不同策略模式，但通过类型注解强制要求它必须是经过验证的 Pydantic 模型）。
    """
    record_id: str
    created_at: str
    prompt_hash: str
    prompt_text: str
    image: BaseModel  # 类型化契约（GeneratedImageRecord 实例），构造前已经验证
    record_schema_version: int = Field(default=1)
    policy_hash: str  # 策略快照哈希，用于审计
    schema_hash: str  # Schema 与策略语义的组合哈希，用于保障迁移安全
    run_id: str  # 运行标识，用于把数据库记录关联到 LangGraph thread_id 和 checkpoint 状态

    def to_json_dict(self) -> dict[str, Any]:
        """通过单一事实来源转换为可写入数据库的 JSON 兼容字典。"""
        # 防御式检查：确保 image 是 BaseModel；虽然 Pydantic 会验证，但显式检查更清晰
        if not isinstance(self.image, BaseModel):
            raise TypeError(f"image must be a BaseModel instance, got {type(self.image)}")

        # 整条记录使用 model_dump，同时显式序列化嵌套的 image 契约
        # 这样可以明确契约边界：这里只序列化已经验证过的 image
        data = self.model_dump(mode="json")
        data["image"] = self.image.model_dump(mode="json")  # 显式进行契约序列化
        return data


class GraphState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    execution_log: Annotated[list[str], add]  # 使用加法 reducer：每个节点都向列表追加内容

    image_prompt: dict[str, Any]
    generated_image: dict[str, Any]

    retry_count: int
    last_error: str
    prompt_ok: bool  # 显式验证状态
    image_ok: bool   # 显式验证状态

    db_record: dict[str, Any]
    active_policy: dict[str, Any]  # 本次运行固定使用的策略
    active_mode: str  # 本次运行的模式快照
    policy_hash: str  # 固定策略的哈希，用作缓存键与审计标识
    schema_hash: str  # Schema 与策略语义的组合哈希，用于保障迁移安全
    max_retries: int  # 从策略中固定的 max_retries，避免重复查找
    run_id: str  # 运行标识，用于把数据库记录关联到 LangGraph thread_id 和 checkpoint 状态


## 构建节点

In [9]:
def node_initialize_policy(state: GraphState):
    """在 START 阶段固定策略，确保整次运行中的验证行为具有确定性。"""
    policy = get_active_policy(cfg)
    policy_dict = OmegaConf.to_container(policy, resolve=True)
    mode = str(cfg.mode)

    # 计算策略哈希，用于缓存与审计
    policy_hash = hash_policy(policy_dict)

    # 预先为该策略构建并缓存模型
    ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord = get_models_for_policy(policy_dict, mode, policy_hash)

    # 提取会影响验证但不会体现在 JSON Schema 中的策略语义
    # 例如 required_keywords、forbid_markdown、require_style_tag
    # 进行充分归一化，以保证哈希稳定
    prompt_cfg = policy_dict["prompt"]
    policy_semantics = {
        "required_keywords": sorted([str(x).lower() for x in prompt_cfg.get("required_keywords", [])]),  # 已归一化
        "forbid_markdown": bool(prompt_cfg.get("forbid_markdown", False)),
        "require_style_tag": bool(prompt_cfg.get("require_style_tag", False)),
        "max_retries": int(policy_dict["max_retries"]),
        # 如果配置中存在 style_tags 列表，也应在这里归一化
        # "style_tags": sorted([str(x).lower() for x in prompt_cfg.get("style_tags", [])]),
    }

    # 计算组合 Schema 哈希：Schema + 策略语义，用于保障迁移安全
    # 即使 JSON Schema 未变，只要验证行为变化，schema_hash 也会变化
    prompt_schema = ImagePrompt.model_json_schema()
    tool_schema = GeneratedImageToolResult.model_json_schema()
    record_schema = GeneratedImageRecord.model_json_schema()
    combined = {
        "schemas": {
            "ImagePrompt": prompt_schema,
            "GeneratedImageToolResult": tool_schema,
            "GeneratedImageRecord": record_schema,
        },
        "policy_semantics": policy_semantics,
    }
    schema_hash = sha256_text(
        json.dumps(combined, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    )[:16]

    # 若 State 中已有 run_id（由配置里的 thread_id 设置）则直接使用，否则生成一个新的
    run_id = state.get("run_id") or f"run_{sha256_text(f'{mode}_{datetime.now(timezone.utc).isoformat()}')[:12]}"

    return {
        "active_policy": policy_dict,
        "active_mode": mode,
        "policy_hash": policy_hash,
        "schema_hash": schema_hash,
        "max_retries": int(policy_dict["max_retries"]),  # 保存到 State 供路由使用，避免重复查找
        "run_id": run_id,  # 数据库记录与 checkpoint 追踪使用的运行标识
        "prompt_ok": False,
        "image_ok": False,
        "execution_log": [f"Initialized: mode={mode}, policy_hash={policy_hash}, schema_hash={schema_hash}, run_id={run_id}"],
    }


def node_content_writer(state: GraphState):
    """智能体 A：撰写内容草稿。"""
    return {
        "execution_log": ["A: Content draft written"],
    }


def node_propose_image_prompt(state: GraphState):
    """智能体 B：提出图像 Prompt；初始结果可能不合法。"""
    return {
        "image_prompt": {
            "prompt": "Short",  # 这个值会触发验证失败
            "size_px": 1024,
        },
        "execution_log": ["B: Proposed image prompt"],
    }


def node_prompt_validation_gate(state: GraphState):
    """验证门：结合重试逻辑执行 Pydantic 契约。"""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    ImagePrompt, _, _ = get_models_for_policy(policy_dict, mode, policy_hash)

    raw_prompt = state.get("image_prompt", {})
    retries = int(state.get("retry_count", 0))
    max_retries = int(state["max_retries"])  # 使用 State 中固定的值

    try:
        valid_prompt = ImagePrompt(**raw_prompt)
        return {
            "image_prompt": valid_prompt.model_dump(),
            "retry_count": 0,
            "last_error": "",
            "prompt_ok": True,  # 显式成功标志
            "execution_log": [f"C: Prompt validated on attempt {retries + 1}"],
        }
    except ValidationError as e:
        error_msg = e.errors()[0]["msg"]

        # 先递增重试次数再判断上限，让日志编号更清晰
        new_retries = retries + 1

        if new_retries > max_retries:
            return {
                "retry_count": new_retries,  # 更新计数，使路由能够识别已超过上限的状态
                "prompt_ok": False,
                "execution_log": [f"C: Max retries ({max_retries}) exceeded on attempt {new_retries}. Last error: {error_msg}"],
                "last_error": error_msg,
            }

        return {
            "retry_count": new_retries,
            "last_error": error_msg,
            "prompt_ok": False,
            "execution_log": [f"C: Validation failed on attempt {new_retries}: {error_msg}"],
        }


def node_refine_prompt(state: GraphState):
    """智能体 D：根据验证错误修正 Prompt。"""
    policy_dict = state["active_policy"]
    last_error = state.get("last_error", "")

    prompt_cfg = policy_dict["prompt"]
    min_len = int(prompt_cfg["min_length"])
    required = [str(x) for x in prompt_cfg.get("required_keywords", [])]
    require_style = bool(prompt_cfg.get("require_style_tag", False))

    # 构造一个满足全部约束的 Prompt
    style_part = "minimal vector illustration" if require_style else ""
    refined_prompt = {
        "prompt": f"A high-resolution technical illustration showing Pydantic validation contracts in a multi-agent system. {style_part} " +
                  " ".join([f"Focus on {kw}." for kw in required]) +
                  f" Clean style. Minimum {min_len} characters.",
        "size_px": 1024,
    }

    return {
        "image_prompt": refined_prompt,
        "execution_log": ["D: Prompt refined based on validation error"],
    }


def node_generate_image(state: GraphState):
    """智能体 E：调用图像生成工具。"""
    prompt = state["image_prompt"]["prompt"]

    tool_output = {
        "asset_id": sha256_text(prompt)[:12],
        "mime_type": "image/png",
        "bytes_size": 150000,
        "width_px": state["image_prompt"]["size_px"],
        "height_px": state["image_prompt"]["size_px"],
    }

    return {
        "generated_image": tool_output,
        "execution_log": ["E: Image generation tool executed"],
    }


def node_image_output_validation_gate(state: GraphState):
    """验证门：在写入数据库前验证工具输出。"""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    _, GeneratedImageToolResult, _ = get_models_for_policy(policy_dict, mode, policy_hash)

    raw_image = state.get("generated_image", {})

    try:
        valid_image = GeneratedImageToolResult(**raw_image)
        return {
            "generated_image": valid_image.model_dump(),
            "image_ok": True,  # 显式成功标志
            "execution_log": ["F: Image tool output validated"],
        }
    except ValidationError as e:
        error_msg = e.errors()[0]["msg"]
        return {
            "generated_image": {},
            "image_ok": False,
            "execution_log": [f"F: Image tool output rejected: {error_msg}"],
        }


def node_prepare_db_record(state: GraphState):
    """准备经过验证并完成 JSON 序列化的数据库记录。"""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    schema_hash = state["schema_hash"]
    _, _, GeneratedImageRecord = get_models_for_policy(policy_dict, mode, policy_hash)

    prompt_text = state["image_prompt"]["prompt"]
    prompt_hash = sha256_text(prompt_text)

    # 把工具结果转换为数据库记录契约；两者可以采用不同验证规则
    # 本例中两者相同，但在生产环境中可以独立演进
    image_record = GeneratedImageRecord(**state["generated_image"])

    record = MediaRecord(
        record_id=f"media_{prompt_hash[:12]}",
        created_at=datetime.now(timezone.utc).isoformat(),
        prompt_hash=prompt_hash,
        prompt_text=prompt_text,
        image=image_record,  # 类型化契约（GeneratedImageRecord BaseModel），而不是字典
        record_schema_version=int(policy_dict["db"]["schema_version"]),
        policy_hash=policy_hash,  # 使用初始化阶段保存的哈希
        schema_hash=schema_hash,  # 使用初始化阶段保存的哈希
        run_id=state["run_id"],  # 用运行标识把数据库记录关联到 thread_id 与 checkpointer
    )

    db_dict = record.to_json_dict()

    return {
        "db_record": db_dict,
        "execution_log": ["G: DB record prepared with JSON serialization"],
    }


def node_persist_to_db(state: GraphState):
    """把经过验证并完成 JSON 序列化的记录持久化到数据库。"""
    record = state["db_record"]
    return {
        "execution_log": [f"H: Persisted to DB: record_id={record['record_id']}"],
    }


def should_retry_prompt(state: GraphState) -> str:
    """根据显式验证状态进行路由。"""
    if state.get("prompt_ok", False):
        return "ok"

    max_retries = int(state["max_retries"])  # 使用 State 中固定的值
    retries = int(state.get("retry_count", 0))

    # 判断是否已超过最大重试次数；节点会先递增，再检查上限
    if retries > max_retries:
        return "end"
    return "fix"


def should_continue_with_image(state: GraphState) -> str:
    """根据显式验证状态进行路由。"""
    return "ok" if state.get("image_ok", False) else "end"


## 构建图（Graph）

In [10]:
memory = InMemorySaver()
workflow = StateGraph(GraphState)

workflow.add_node("init_policy", node_initialize_policy)
workflow.add_node("writer", node_content_writer)
workflow.add_node("propose_prompt", node_propose_image_prompt)
workflow.add_node("prompt_gate", node_prompt_validation_gate)
workflow.add_node("refine_prompt", node_refine_prompt)
workflow.add_node("generate_image", node_generate_image)
workflow.add_node("image_gate", node_image_output_validation_gate)
workflow.add_node("prepare_db", node_prepare_db_record)
workflow.add_node("persist_db", node_persist_to_db)

workflow.add_edge(START, "init_policy")
workflow.add_edge("init_policy", "writer")
workflow.add_edge("writer", "propose_prompt")
workflow.add_edge("propose_prompt", "prompt_gate")

workflow.add_conditional_edges(
    "prompt_gate",
    should_retry_prompt,
    {"fix": "refine_prompt", "ok": "generate_image", "end": END},
)
workflow.add_edge("refine_prompt", "prompt_gate")

workflow.add_edge("generate_image", "image_gate")
workflow.add_conditional_edges(
    "image_gate",
    should_continue_with_image,
    {"ok": "prepare_db", "end": END},
)

workflow.add_edge("prepare_db", "persist_db")
workflow.add_edge("persist_db", END)

app = workflow.compile(checkpointer=memory)


## 运行应用

In [11]:
print("=" * 80)
print("PRODUCTION SYSTEM DEMONSTRATION")
print("=" * 80)
# 注意：这里为了初始展示从 cfg 读取；真正执行时使用 State 中固定的值
policy = get_active_policy(cfg)
print(f"Initial Mode: {cfg.mode}")
print(f"Max Retries: {policy.max_retries}")
print(f"Prompt Min Length: {policy.prompt.min_length}")
print(f"Required Keywords: {policy.prompt.required_keywords}")
print(f"Forbid Markdown: {policy.prompt.forbid_markdown}")
print(f"Require Style Tag: {policy.prompt.require_style_tag}")
print("=" * 80)
print()

config = {"configurable": {"thread_id": "production_run_001"}}

# 从 thread_id 设置 run_id，把数据库记录关联到 LangGraph checkpoint 状态
initial_state: GraphState = {
    "messages": [HumanMessage(content="Generate a blog cover image")],
    "execution_log": [],
    "image_prompt": {},
    "generated_image": {},
    "retry_count": 0,
    "last_error": "",
    "prompt_ok": False,
    "image_ok": False,
    "db_record": {},
    "active_policy": {},
    "active_mode": "",
    "policy_hash": "",
    "schema_hash": "",
    "max_retries": 0,
    "run_id": config["configurable"]["thread_id"],  # 把数据库记录关联到 LangGraph thread_id
}

print("--- Running Complete Workflow ---")
result = app.invoke(initial_state, config)

print("\n--- Execution Log ---")
for line in result["execution_log"]:
    print(f"  {line}")

print("\n--- DB Record (JSON-Compatible with Versioning) ---")
print(json.dumps(result["db_record"], indent=2, ensure_ascii=False))

print("\n--- Policy Snapshot (Pinned at Start) ---")
print(f"  Mode: {result['active_mode']}")  # 使用 State 中固定的值，而不是 cfg.mode
print(f"  Max Retries: {result.get('max_retries', 'N/A')}")  # Use pinned value from state
print(f"  Run ID: {result.get('run_id', 'N/A')}")  # 用于追踪的运行标识
print(f"  Policy Hash: {result['db_record'].get('policy_hash', 'N/A')}")
print(f"  Schema Hash: {result['db_record'].get('schema_hash', 'N/A')}")
print(f"  Schema Version: {result['db_record'].get('record_schema_version', 'N/A')}")

print("\n--- Fault Tolerance: Checkpoint Inspection ---")
snapshot = app.get_state(config)
print(f"  Last node: {snapshot.next if hasattr(snapshot, 'next') else 'N/A'}")
print(f"  Policy pinned: {bool(snapshot.values.get('active_policy'))}")
print(f"  Run ID: {snapshot.values.get('run_id', 'N/A')}")  # 把数据库记录关联到 checkpoint
print(f"  Policy hash: {snapshot.values.get('policy_hash', 'N/A')}")
print(f"  Schema hash: {snapshot.values.get('schema_hash', 'N/A')}")

PRODUCTION SYSTEM DEMONSTRATION
Initial Mode: strict
Max Retries: 3
Prompt Min Length: 50
Required Keywords: ['pydantic', 'validation', 'contract']
Forbid Markdown: True
Require Style Tag: True

--- Running Complete Workflow ---

--- Execution Log ---
  Initialized: mode=strict, policy_hash=64a35d3ba43027ae, schema_hash=7720b7473856aab3, run_id=production_run_001
  A: Content draft written
  B: Proposed image prompt
  C: Validation failed on attempt 1: String should have at least 50 characters
  D: Prompt refined based on validation error
  C: Prompt validated on attempt 2
  E: Image generation tool executed
  F: Image tool output validated
  G: DB record prepared with JSON serialization
  H: Persisted to DB: record_id=media_ad4f8f9896c2

--- DB Record (JSON-Compatible with Versioning) ---
{
  "record_id": "media_ad4f8f9896c2",
  "created_at": "2026-06-10T12:39:15.804294+00:00",
  "prompt_hash": "ad4f8f9896c2184e5f2a7861d1eac320a0927a12753e12d489397a324a8d90ae",
  "prompt_text": "A hig

## Pydantic 与 Instructor：边界保护 vs. 源头恢复

在这个 LangGraph 工作流中，Pydantic 扮演的是边界验证门（boundary gate）。它可以阻止无效数据继续传播，但恢复逻辑仍然存在于编排层，例如独立的 refine 节点以及显式的重试路由。

当你希望把恢复机制进一步推近生成源头时，Instructor 会非常有用。它不再把结构化输出验证视为独立的工作流问题，而是把 Schema 约束与生成过程绑定在一起：模型会基于同一套固定的 Pydantic 契约反复尝试生成合格输出，并把验证错误作为反馈。这样可以减少图中的分支、去掉特定的“修复器（fixer）”节点，并让不同模型与提供商之间的结构化输出行为更加一致。

换句话说，Pydantic 负责保护系统边界；Instructor 则减少稳定命中这道边界所需要的额外工作。

# Instructor：在生成源头恢复（Recovery at the Source）

### 复用前面相同的 Pydantic 模型（沿用策略驱动的模型）

In [13]:
policy = get_active_policy(cfg)
policy_dict = OmegaConf.to_container(policy, resolve=True)
mode = str(cfg.mode)
policy_hash = hash_policy(policy_dict)

ImagePrompt, _, _ = get_models_for_policy(policy_dict, mode, policy_hash)

# 创建 Instructor Client；底层使用兼容 OpenRouter 的 OpenAI Client
# Instructor 通过 patch 提供商的 create() 方法工作
# 架构：Pydantic Model → Provider Handler → Dispatcher → Provider Client → Retry Layer → Instructor（patched）
from openai import OpenAI
model_slug = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-haiku")

# 创建面向 OpenRouter 配置的 OpenAI Client
client = instructor.from_provider(
    f"openrouter/{model_slug}",
    base_url="https://openrouter.ai/api/v1",
    mode=instructor.Mode.TOOLS,
)

print("=" * 80)
print("Automatic Recovery at Generation Time")
print("=" * 80)
print(f"Policy Mode: {mode}")
print(f"Max Retries: {policy.max_retries}")
print(f"Prompt Min Length: {policy.prompt.min_length}")
print(f"Required Keywords: {policy.prompt.required_keywords}")
print("=" * 80)
print()

# Instructor 会携带验证错误反馈自动重试
# 不再需要独立的 refine 节点或条件重试路由
print("--- Generating Image Prompt with Instructor ---")
print("Instructor will automatically retry if validation fails, using error messages as feedback.")
print()

try:
    # Instructor 在内部完成验证，不需要手工编写重试逻辑
    # patch 后的 Client 支持 response_model 与 max_retries
    # 流程：create() → retry layer → process_response → validation → 必要时重新提问
    # 验证失败时，Instructor 的 handle_reask_kwargs() 会追加错误反馈
    result = client.create(                       # 原调用形式为 client.chat.completions.create(...)
        messages=[
            {"role": "user",
            "content": "Generate an image prompt for a blog cover about Pydantic validation in multi-agent systems."}
        ],
        response_model=ImagePrompt,               # 触发 Schema / Tool 绑定与解析
        max_retries=policy.max_retries,           # 基于 tenacity 的验证重试
        temperature=0.2,
    )

    # Instructor 直接返回解析后的 Pydantic 模型，并附带 _raw_response
    print(f"✓ Successfully generated valid prompt after automatic retries:")
    print(f"  Prompt: {result.prompt[:100]}...")
    print(f"  Length: {len(result.prompt)} characters")
    print(f"  Size: {result.size_px}px")
    # 如有需要，可通过 result._raw_response 访问提供商原始响应
    print()
    print("Key difference: No separate 'refine_prompt' node needed!")
    print("Instructor handled validation and retry logic internally.")

except Exception as e:
    print(f"✗ Failed after max retries: {e}")



Automatic Recovery at Generation Time
Policy Mode: strict
Max Retries: 3
Prompt Min Length: 50
Required Keywords: ['pydantic', 'validation', 'contract']

--- Generating Image Prompt with Instructor ---
Instructor will automatically retry if validation fails, using error messages as feedback.

✓ Successfully generated valid prompt after automatic retries:
  Prompt: A futuristic digital landscape showcasing a contract-based validation system for multi-agent AI inte...
  Length: 568 characters
  Size: 1024px

Key difference: No separate 'refine_prompt' node needed!
Instructor handled validation and retry logic internally.


# 使用 Instructor 简化 LangGraph 工作流

In [16]:
# 简化后的 State：不再需要 retry_count、last_error 或 prompt_ok
class SimplifiedGraphState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    execution_log: Annotated[list[str], add]
    image_prompt: dict[str, Any]
    generated_image: dict[str, Any]

def node_propose_prompt_with_instructor(state: SimplifiedGraphState):
    """使用 Instructor 生成 Prompt；验证与重试都在这里发生。"""
    # Instructor 在内部完成验证，因此这里可以直接拿到合法 Prompt
    # patch 后的 Client 会携带验证错误反馈自动重试
    result = client.chat.completions.create(
        model=os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-haiku"),
        messages=[
            {
                "role": "user",
                "content": "Generate an image prompt for a blog cover about Pydantic validation in multi-agent systems."
            }
        ],
        response_model=ImagePrompt,
        max_retries=policy.max_retries,
        temperature=0.2,
    )

    return {
        "image_prompt": result.model_dump(),
        "execution_log": ["B: Generated valid prompt with Instructor (automatic retry on validation errors)"],
    }

def node_generate_image_simple(state: SimplifiedGraphState):
    """生成图像；如果 Instructor 已保证输入合法，就无需额外输入验证门。"""
    prompt = state["image_prompt"]["prompt"]

    tool_output = {
        "asset_id": sha256_text(prompt)[:12],
        "mime_type": "image/png",
        "bytes_size": 150000,
        "width_px": state["image_prompt"]["size_px"],
        "height_px": state["image_prompt"]["size_px"],
    }

    return {
        "generated_image": tool_output,
        "execution_log": ["E: Image generation tool executed"],
    }

# 构建简化后的图：没有 refine 节点，也没有条件重试路由
simplified_workflow = StateGraph(SimplifiedGraphState)

simplified_workflow.add_node("propose_prompt", node_propose_prompt_with_instructor)
simplified_workflow.add_node("generate_image", node_generate_image_simple)

# 简单线性流程：无需为重试引入分支
simplified_workflow.add_edge(START, "propose_prompt")
simplified_workflow.add_edge("propose_prompt", "generate_image")
simplified_workflow.add_edge("generate_image", END)

simplified_app = simplified_workflow.compile(checkpointer=MemorySaver())

print("=" * 80)
print("Running Simplified LangGraph Workflow with Instructor")
print("=" * 80)
print()

initial_state = {
    "messages": [HumanMessage(content="Generate a blog cover image")],
    "execution_log": [],
    "image_prompt": {},
    "generated_image": {},
}

result = simplified_app.invoke(initial_state, {"configurable": {"thread_id": "instructor_demo"}})

print("--- Execution Log ---")
for line in result["execution_log"]:
    print(f"  {line}")

print()
print("--- Generated Prompt ---")
print(f"  Prompt: {result['image_prompt'].get('prompt', 'N/A')[:100]}...")
print(f"  Length: {len(result['image_prompt'].get('prompt', ''))} characters")

print()
print("=" * 80)
print("GRAPH COMPARISON:")
print("=" * 80)
print("Previous (LangGraph + Pydantic):")
print("  START → init_policy → writer → propose_prompt → prompt_gate")
print("  prompt_gate → [fix: refine_prompt → prompt_gate] | [ok: generate_image]")
print("  generate_image → image_gate → [ok: prepare_db] | [end: END]")
print("  prepare_db → persist_db → END")
print()
print("With Instructor:")
print("  START → propose_prompt → generate_image → END")
print()
print("Nodes removed: init_policy, writer, prompt_gate, refine_prompt,")
print("               image_gate, prepare_db, persist_db")
print("Conditional edges removed: retry routing logic")
print("=" * 80)

Running Simplified LangGraph Workflow with Instructor

--- Execution Log ---
  B: Generated valid prompt with Instructor (automatic retry on validation errors)
  E: Image generation tool executed

--- Generated Prompt ---
  Prompt: A futuristic digital landscape [vector] showcasing a data contract validation process in a multi-age...
  Length: 456 characters

GRAPH COMPARISON:
Previous (LangGraph + Pydantic):
  START → init_policy → writer → propose_prompt → prompt_gate
  prompt_gate → [fix: refine_prompt → prompt_gate] | [ok: generate_image]
  generate_image → image_gate → [ok: prepare_db] | [end: END]
  prepare_db → persist_db → END

With Instructor:
  START → propose_prompt → generate_image → END

Nodes removed: init_policy, writer, prompt_gate, refine_prompt,
               image_gate, prepare_db, persist_db
Conditional edges removed: retry routing logic
